In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

login(secret_value_0)
print("Authenncation.. sucessful")

Authenncation.. sucessful


In [3]:
# ============================================================
# N-ATLaS Language Foundry
# Notebook: 04_tiv_parallel_corpus_audit.ipynb
# Checkpoint 1: Environment and source configuration
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import json
import platform

import pandas as pd
from datasets import load_dataset

EXPERIMENT_CONFIG = {
    "experiment_id": "tiv-translation-pilot-v1",
    "target_language_code": "tiv",
    "source_language_code": "en",
    "random_seed": 42,
    "train_ratio": 0.75,
    "validation_ratio": 0.10,
    "test_ratio": 0.15,
}

SOURCES = {
    "mt560": {
        "dataset_id": (
            "michsethowusu/"
            "english-tiv_sentence-pairs_mt560"
        ),
        "split": "train",
        "license": "CC-BY-4.0",
        "expected_fields": ["eng", "tiv"],
    },
    "hypa": {
        "dataset_id": "hypaai/Hypa-Text-10k",
        "split": "train",
        "license": "Apache-2.0",
        "expected_fields": ["messages"],
    },
}

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Experiment configuration:")
print(json.dumps(EXPERIMENT_CONFIG, indent=2))
print("\nSource configuration:")
print(json.dumps(SOURCES, indent=2))
print("\nAudit started at:", datetime.now(timezone.utc).isoformat())

Python: 3.13.15
Pandas: 2.3.3
Experiment configuration:
{
  "experiment_id": "tiv-translation-pilot-v1",
  "target_language_code": "tiv",
  "source_language_code": "en",
  "random_seed": 42,
  "train_ratio": 0.75,
  "validation_ratio": 0.1,
  "test_ratio": 0.15
}

Source configuration:
{
  "mt560": {
    "dataset_id": "michsethowusu/english-tiv_sentence-pairs_mt560",
    "split": "train",
    "license": "CC-BY-4.0",
    "expected_fields": [
      "eng",
      "tiv"
    ]
  },
  "hypa": {
    "dataset_id": "hypaai/Hypa-Text-10k",
    "split": "train",
    "license": "Apache-2.0",
    "expected_fields": [
      "messages"
    ]
  }
}

Audit started at: 2026-10-10T19:04:32.099089+00:00


In [4]:
# ============================================================
# Checkpoint 2: Load source datasets
# ============================================================

mt560_ds = load_dataset(
    SOURCES["mt560"]["dataset_id"],
    split=SOURCES["mt560"]["split"],
)

hypa_ds = load_dataset(
    SOURCES["hypa"]["dataset_id"],
    split=SOURCES["hypa"]["split"],
)

print("MT560 rows:", len(mt560_ds))
print("MT560 columns:", mt560_ds.column_names)

print("\nHypa rows:", len(hypa_ds))
print("Hypa columns:", hypa_ds.column_names)

README.md:   0%|          | 0.00/843 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/184113 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/18.8k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.32MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

MT560 rows: 184113
MT560 columns: ['eng', 'tiv']

Hypa rows: 10000
Hypa columns: ['messages', 'tools_json']


In [5]:
# ============================================================
# Checkpoint 3A: Inspect MT560
# ============================================================

mt560_df = mt560_ds.to_pandas()

display(mt560_df.head(10))

print("\nColumn types:")
print(mt560_df.dtypes)

print("\nMissing values:")
print(mt560_df[["eng", "tiv"]].isna().sum())

print("\nExact duplicate pairs:")
print(
    mt560_df.duplicated(subset=["eng", "tiv"]).sum()
)

,eng,tiv
0,Prove Yourselves Ready,Kohol Nen Iyol
1,I Shall Certainly Sanctify My Great Name,Me Tsegha Iti Yam i Vesen
2,He Is Useful to Me,Ngum a Inja Kpishi Je
3,He Who Hates Bribes Will Live,"Or u Nan Vende Inyaregh Ki Yamen Ishô la yô , ..."
4,""" Continue . . . to do good . "" - LUKE 6 : 35 .",""" Eren nen . . . a [ ior ] doo doo . "" - LUKA ..."
5,"In this case , the "" heart "" relates to one 's...",""" Ishima "" i i ter i heen ne tile sha ityough ..."
6,"What has "" the day of salvation "" meant for an...",""" Iyange i myom "" la ngi a inja hen Mbakristu ..."
7,""" But keep getting filled with spirit . "" - 5 ...",""" Kpa iv nen a Jijingi . "" - 5 : 18 ."
8,""" I have made your name known to them . "" - Jo...",""" M pase ve iti You . "" - Yohane 17 : 26"
9,""" I will also bring them to my holy mountain a...",""" M va a ve hen icighanwo Wam je , Me na , á s..."



Column types:
eng    object
tiv    object
dtype: object

Missing values:
eng    0
tiv    0
dtype: int64

Exact duplicate pairs:
14


In [6]:
# ============================================================
# Checkpoint 3B: Inspect Hypa message structure
# ============================================================

print("Hypa sample record keys:")
print(hypa_ds[0].keys())

for index in range(min(5, len(hypa_ds))):
    row = hypa_ds[index]

    print(f"\n{'=' * 70}")
    print(f"RECORD {index}")

    messages = row.get("messages", [])

    for message in messages:
        role = message.get("role", "unknown")
        content = message.get("content", "")

        if role not in {"system", "user", "assistant"}:
            continue

        # Avoid dumping very long messages into notebook output.
        if isinstance(content, str):
            content = content[:700]

        print(f"\n[{role.upper()}]")
        print(content)

Hypa sample record keys:
dict_keys(['messages', 'tools_json'])

RECORD 0

[SYSTEM]
You are Hypa Translate. Translate from Urhobo to English. Return only the exact translation.

THINKING=None

[USER]
Ruth rọvwọn re, o ji vwo emọ ẹnẹ.

[ASSISTANT]
Ruth is married with four children.

RECORD 1

[SYSTEM]
You are a helpful and honest multilingual translation expert specialized in translating to English.

[USER]
Translate this to English and Return ONLY THE EXACT TRANSLATION. "Ikó ami 'nominal nómba' ikpeneke eke etóngó ekama njen agwo inekeke isóp ikama anye."

[ASSISTANT]
The term "nominal number" is quite recent and of limited use.

RECORD 2

[SYSTEM]
You are Hypa Translate. Translate from Ebira to Idoma. Preserve meaning faithfully and output only the final translation with no explanation, no notes, and no extra text.

THINKING=None

[USER]
Òsòónì ò si hiètè íjòvà ànà tite chengedee dò si ùkata òwòwa ngwu ni ùmè íjòvà i Ming.

[ASSISTANT]
Enya l'igomenti olohi ya o ke l'ofu empe biwa ipu

In [7]:
# ============================================================
# Checkpoint 4: Discover Hypa translation directions
# ============================================================

import re
from collections import Counter

def get_message_content(message):
    """Extract readable message content without assuming its type."""
    content = message.get("content", "")

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("text"):
                parts.append(str(item["text"]))
            elif isinstance(item, str):
                parts.append(item)
        return " ".join(parts).strip()

    return ""


hypa_audit_rows = []

for index, row in enumerate(hypa_ds):
    messages = row.get("messages", [])

    system_parts = []
    user_parts = []
    assistant_parts = []

    for message in messages:
        role = str(message.get("role", "")).lower()
        content = get_message_content(message)

        if role == "system":
            system_parts.append(content)
        elif role == "user":
            user_parts.append(content)
        elif role == "assistant":
            assistant_parts.append(content)

    system_text = " ".join(system_parts)

    hypa_audit_rows.append({
        "row_index": index,
        "system": system_text,
        "user": " ".join(user_parts),
        "assistant": " ".join(assistant_parts),
    })

hypa_audit_df = pd.DataFrame(hypa_audit_rows)

# Identify instructions mentioning Tiv, English, or both.
system_lower = hypa_audit_df["system"].str.lower()

mentions_tiv = system_lower.str.contains(
    r"\btiv\b", regex=True, na=False
)

mentions_english = system_lower.str.contains(
    r"\benglish\b", regex=True, na=False
)

print("Total Hypa rows:", len(hypa_audit_df))
print("Instructions mentioning Tiv:", int(mentions_tiv.sum()))
print("Instructions mentioning English:", int(mentions_english.sum()))
print(
    "Instructions mentioning both:",
    int((mentions_tiv & mentions_english).sum()),
)

# Count distinct instructions that mention either language.
relevant = hypa_audit_df[mentions_tiv | mentions_english]

instruction_counts = (
    relevant.groupby("system", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

display(instruction_counts.head(100))

Total Hypa rows: 10000
Instructions mentioning Tiv: 1639
Instructions mentioning English: 1606
Instructions mentioning both: 152


,system,row_count
130,You are Hypa Translate. Translate to English. ...,273
133,You are a helpful and honest multilingual tran...,203
29,You are Hypa Translate. Correct the below tran...,89
39,You are Hypa Translate. Translate from Efik to...,65
132,You are Hypa Translate. Translate to Tiv. Retu...,59
...,...,...
71,You are Hypa Translate. Translate from English...,9
9,"You are Hypa Dictionary, a English dictionary....",9
77,You are Hypa Translate. Translate from Hausa t...,9
4,You are Hypa Dictionary API. For the given wor...,8


In [8]:
# ============================================================
# Checkpoint 5: Inspect English/Tiv candidate instructions
# ============================================================

candidate_instructions = hypa_audit_df[
    mentions_tiv & mentions_english
].copy()

print(
    "Rows with instructions mentioning both languages:",
    len(candidate_instructions),
)

for _, record in candidate_instructions.head(30).iterrows():
    print("\n" + "=" * 80)
    print("ROW INDEX:", record["row_index"])
    print("SYSTEM:", record["system"])
    print("USER:", record["user"][:500])
    print("ASSISTANT:", record["assistant"][:500])

Rows with instructions mentioning both languages: 152

ROW INDEX: 81
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: He was one of the defenders of the Rouen School.
ASSISTANT: Un kpa lu mom ken mbakuran mba Rouen School.

ROW INDEX: 265
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: She has to keep begging him, Will you keep me one more day?
ASSISTANT: "Gba u una za hemen u zamber a na, u za hemen u luu a mo iyange igen kpa?"

ROW INDEX: 304
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: Many species are day-flying with metallic patterns.
ASSISTANT: Atô a akaa a uma nga pulugh atetan man mba a mlu u myôm.

ROW INDEX: 416
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: In the Netherlands, officers of arms

In [9]:
# ============================================================
# Checkpoint 6: Classify Hypa translation directions
# ============================================================

def classify_hypa_instruction(system_text):
    text = str(system_text).lower()
    text = re.sub(r"\s+", " ", text).strip()

    # Explicit directions first.
    if re.search(
        r"translate\s+from\s+english\s+to\s+tiv\b", text
    ):
        return "explicit_en_to_tiv"

    if re.search(
        r"translate\s+from\s+tiv\s+to\s+english\b", text
    ):
        return "explicit_tiv_to_en"

    # Target language is specified, but source may not be.
    if re.search(r"translate\s+to\s+tiv\b", text):
        return "target_tiv_source_unspecified"

    if re.search(r"translate\s+to\s+english\b", text):
        return "target_english_source_unspecified"

    # General instructions that mention both languages without
    # clearly specifying the translation direction.
    if (
        re.search(r"\benglish\b", text)
        and re.search(r"\btiv\b", text)
        and re.search(r"\btranslat", text)
    ):
        return "mentions_both_direction_unclear"

    if re.search(r"\btiv\b", text):
        return "mentions_tiv_other_task_or_direction"

    return "other_or_unclassified"


hypa_audit_df["instruction_class"] = (
    hypa_audit_df["system"].apply(classify_hypa_instruction)
)

class_summary = (
    hypa_audit_df["instruction_class"]
    .value_counts()
    .rename_axis("instruction_class")
    .reset_index(name="row_count")
)

display(class_summary)

print("\n===== DISTINCT SYSTEM INSTRUCTIONS FOR TIV-RELATED CLASSES =====")

tiv_classes = [
    "explicit_en_to_tiv",
    "explicit_tiv_to_en",
    "target_tiv_source_unspecified",
    "mentions_both_direction_unclear",
    "mentions_tiv_other_task_or_direction",
]

tiv_related = hypa_audit_df[
    hypa_audit_df["instruction_class"].isin(tiv_classes)
]

display(
    tiv_related.groupby(
        ["instruction_class", "system"],
        dropna=False,
    )
    .size()
    .reset_index(name="row_count")
    .sort_values(
        ["instruction_class", "row_count"],
        ascending=[True, False],
    )
    .head(100)
)

print("\n===== EXPLICIT ENGLISH-TO-TIV RECORDS =====")

explicit_candidates = hypa_audit_df[
    hypa_audit_df["instruction_class"] == "explicit_en_to_tiv"
]

print("Explicit English-to-Tiv records:", len(explicit_candidates))

for _, record in explicit_candidates.head(10).iterrows():
    print("\n" + "=" * 70)
    print("ROW INDEX:", record["row_index"])
    print("SYSTEM:", record["system"])
    print("USER:", record["user"][:400])
    print("ASSISTANT:", record["assistant"][:400])

,instruction_class,row_count
0,other_or_unclassified,8088
1,mentions_tiv_other_task_or_direction,1385
2,target_english_source_unspecified,273
3,explicit_en_to_tiv,103
4,target_tiv_source_unspecified,102
5,explicit_tiv_to_en,49



===== DISTINCT SYSTEM INSTRUCTIONS FOR TIV-RELATED CLASSES =====


,instruction_class,system,row_count
1,explicit_en_to_tiv,You are Hypa Translate. Translate from English...,56
0,explicit_en_to_tiv,You are Hypa Translate. Translate from English...,47
2,explicit_tiv_to_en,You are Hypa Translate. Translate from Tiv to ...,49
9,mentions_tiv_other_task_or_direction,You are Hypa Translate. Correct the below tran...,89
16,mentions_tiv_other_task_or_direction,You are Hypa Translate. Translate from Efik to...,65
...,...,...,...
58,mentions_tiv_other_task_or_direction,You are Hypa Translate. Translate from Tiv to ...,3
57,mentions_tiv_other_task_or_direction,You are Hypa Translate. Translate from Tiv to ...,2
39,mentions_tiv_other_task_or_direction,You are Hypa Translate. Translate from Tiv to ...,1
67,target_tiv_source_unspecified,You are Hypa Translate. Translate to Tiv. Retu...,59



===== EXPLICIT ENGLISH-TO-TIV RECORDS =====
Explicit English-to-Tiv records: 103

ROW INDEX: 81
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: He was one of the defenders of the Rouen School.
ASSISTANT: Un kpa lu mom ken mbakuran mba Rouen School.

ROW INDEX: 265
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: She has to keep begging him, Will you keep me one more day?
ASSISTANT: "Gba u una za hemen u zamber a na, u za hemen u luu a mo iyange igen kpa?"

ROW INDEX: 304
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: Many species are day-flying with metallic patterns.
ASSISTANT: Atô a akaa a uma nga pulugh atetan man mba a mlu u myôm.

ROW INDEX: 416
SYSTEM: You are Hypa Translate. Translate from English to Tiv. Return only the exact translation.

THINKING=None
USER: In the N

In [10]:
# ============================================================
# Checkpoint 7: Audit the 103 explicit English-to-Tiv candidates
# ============================================================

import hashlib
import re
import pandas as pd

def normalize_for_comparison(text):
    """Normalize whitespace and case for duplicate detection."""
    return re.sub(r"\s+", " ", str(text)).strip().casefold()


# Select only records with explicit English-to-Tiv instructions.
hypa_en_tiv_candidates = (
    hypa_audit_df.loc[
        hypa_audit_df["instruction_class"] == "explicit_en_to_tiv"
    ]
    .copy()
    .reset_index(drop=True)
)

audit_rows = []

for _, row in hypa_en_tiv_candidates.iterrows():
    english = row["user"]
    tiv = row["assistant"]

    # Preserve source index so every candidate can be traced back
    # to its original record in Hypa.
    audit_rows.append({
        "source": "hypa-text-10k",
        "source_row_index": int(row["row_index"]),
        "system_instruction": row["system"],
        "english": english,
        "tiv": tiv,
        "english_empty": not bool(english.strip()),
        "tiv_empty": not bool(tiv.strip()),
        "english_chars": len(english.strip()),
        "tiv_chars": len(tiv.strip()),
        "english_normalized": normalize_for_comparison(english),
        "tiv_normalized": normalize_for_comparison(tiv),
    })

hypa_en_tiv_audit = pd.DataFrame(audit_rows)

# Structural checks.
hypa_en_tiv_audit["missing_text"] = (
    hypa_en_tiv_audit["english_empty"]
    | hypa_en_tiv_audit["tiv_empty"]
)

hypa_en_tiv_audit["duplicate_pair"] = (
    hypa_en_tiv_audit.duplicated(
        subset=["english_normalized", "tiv_normalized"],
        keep="first",
    )
)

hypa_en_tiv_audit["duplicate_english_input"] = (
    hypa_en_tiv_audit.duplicated(
        subset=["english_normalized"],
        keep="first",
    )
)

# Conservative formatting flags for manual inspection.
# These flags do not automatically reject a translation.
hypa_en_tiv_audit["format_warning"] = (
    hypa_en_tiv_audit["english"].str.contains(
        r"^\s*(?:SYSTEM:|USER:|ASSISTANT:)|"
        r"<\|[^|]+\|>|^\s*```",
        case=False,
        regex=True,
        na=False,
    )
    |
    hypa_en_tiv_audit["tiv"].str.contains(
        r"^\s*(?:SYSTEM:|USER:|ASSISTANT:)|"
        r"<\|[^|]+\|>|^\s*```",
        case=False,
        regex=True,
        na=False,
    )
)

print("===== H Y P A  S O U R C E  A U D I T =====")
print("Explicit English-to-Tiv candidates:", len(hypa_en_tiv_audit))
print("Missing or empty text:", int(hypa_en_tiv_audit["missing_text"].sum()))
print("Duplicate English-Tiv pairs:", int(hypa_en_tiv_audit["duplicate_pair"].sum()))
print("Repeated English inputs:", int(hypa_en_tiv_audit["duplicate_english_input"].sum()))
print("Formatting warnings:", int(hypa_en_tiv_audit["format_warning"].sum()))

print("\n===== TEXT LENGTH SUMMARY =====")
display(
    hypa_en_tiv_audit[["english_chars", "tiv_chars"]].describe().round(2)
)

print("\n===== RECORDS REQUIRING STRUCTURAL REVIEW =====")
review_mask = (
    hypa_en_tiv_audit["missing_text"]
    | hypa_en_tiv_audit["duplicate_pair"]
    | hypa_en_tiv_audit["format_warning"]
)

display(
    hypa_en_tiv_audit.loc[
        review_mask,
        [
            "source_row_index",
            "english",
            "tiv",
            "missing_text",
            "duplicate_pair",
            "format_warning",
        ],
    ].head(30)
)

print("\n===== SAMPLE OF AUDITED PAIRS =====")
display(
    hypa_en_tiv_audit[
        [
            "source_row_index",
            "english",
            "tiv",
            "english_chars",
            "tiv_chars",
        ]
    ].head(15)
)

===== H Y P A  S O U R C E  A U D I T =====
Explicit English-to-Tiv candidates: 103
Missing or empty text: 0
Duplicate English-Tiv pairs: 0
Repeated English inputs: 0
Formatting warnings: 0

===== TEXT LENGTH SUMMARY =====


,english_chars,tiv_chars
count,103.00,103.00
mean,77.83,89.01
std,86.02,100.79
min,7.00,24.00
25%,48.00,55.50
50%,64.00,74.00
75%,80.50,97.00
max,669.00,802.00



===== RECORDS REQUIRING STRUCTURAL REVIEW =====


,source_row_index,english,tiv,missing_text,duplicate_pair,format_warning



===== SAMPLE OF AUDITED PAIRS =====


,source_row_index,english,tiv,english_chars,tiv_chars
0,81,He was one of the defenders of the Rouen School.,Un kpa lu mom ken mbakuran mba Rouen School.,48,44
1,265,"She has to keep begging him, Will you keep me ...","""Gba u una za hemen u zamber a na, u za hemen ...",59,74
2,304,Many species are day-flying with metallic patt...,Atô a akaa a uma nga pulugh atetan man mba a m...,51,56
3,416,"In the Netherlands, officers of arms do not ex...","Ken tar u Netherlands yô, mbatomov mba utya mb...",73,69
4,436,He was also a member of Air Canada's board of ...,Shi yange lu or u ken mbahemenev mba Air Canada.,56,48
5,521,Under the scoreboard was a refreshment stall w...,Yange ijiir i teen kwaghyaan lu vegher u shin ...,85,120
6,542,Talbot himself turned down a move to Italian s...,Talbot iyol na yange venda u yemen ken vegher ...,83,115
7,642,He became a wealthy landowner of several thous...,Yange hingir shagba or u lun a inya kpishi kum...,81,97
8,671,### Antonyms:\n- Smooth \n- Unmarked \n- Pla...,### Asember a a lu kposo kposo la:\n- Lue-lue\...,68,109
9,708,That would be very early for him to have been ...,Ngula man i lu tsuaa yum u wen lun or ngeren k...,57,56


In [11]:

# Full audit summary for the 103 explicit English-to-Tiv candidates

summary = {
    "total_candidates": len(hypa_en_tiv_audit),
    "missing_or_empty_text": int(
        hypa_en_tiv_audit["missing_text"].sum()
    ),
    "duplicate_english_tiv_pairs": int(
        hypa_en_tiv_audit["duplicate_pair"].sum()
    ),
    "repeated_english_inputs": int(
        hypa_en_tiv_audit["duplicate_english_input"].sum()
    ),
    "formatting_warnings": int(
        hypa_en_tiv_audit["format_warning"].sum()
    ),
}

print("H Y P A   A U D I T   S U M M A R Y")
print("=" * 42)

for metric, count in summary.items():
    print(f"{metric}: {count}")

# Review all records flagged by any structural check.
review_mask = (
    hypa_en_tiv_audit["missing_text"]
    | hypa_en_tiv_audit["duplicate_pair"]
    | hypa_en_tiv_audit["format_warning"]
)

flagged_records = hypa_en_tiv_audit.loc[
    review_mask,
    [
        "source_row_index",
        "english",
        "tiv",
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "format_warning",
    ],
].copy()

print("\nFLAGGED RECORDS")
print("=" * 42)
print("Total flagged:", len(flagged_records))

display(flagged_records)

print("\nTEXT LENGTH STATISTICS")
display(
    hypa_en_tiv_audit[
        ["english_chars", "tiv_chars"]
    ].describe().round(2)
)


H Y P A   A U D I T   S U M M A R Y
total_candidates: 103
missing_or_empty_text: 0
duplicate_english_tiv_pairs: 0
repeated_english_inputs: 0
formatting_warnings: 0

FLAGGED RECORDS
Total flagged: 0


,source_row_index,english,tiv,missing_text,duplicate_pair,duplicate_english_input,format_warning



TEXT LENGTH STATISTICS


,english_chars,tiv_chars
count,103.00,103.00
mean,77.83,89.01
std,86.02,100.79
min,7.00,24.00
25%,48.00,55.50
50%,64.00,74.00
75%,80.50,97.00
max,669.00,802.00


In [12]:
# Inspect potential truncated  formatted exmaples..

review_indices = [265, 671, 908]

for source_index in review_indices:
    matches = hypa_en_tiv_audit[
        hypa_en_tiv_audit["source_row_index"] == source_index
    ]

    if matches.empty:
        continue

    record = matches.iloc[0]

    print(f"\n{'=' * 70}")
    print(f"Source row: {source_index}")
    print("\nENGLISH:")
    print(record["english"])
    print("\nTIV:")
    print(record["tiv"])



Source row: 265

ENGLISH:
She has to keep begging him, Will you keep me one more day?

TIV:
"Gba u una za hemen u zamber a na, u za hemen u luu a mo iyange igen kpa?"

Source row: 671

ENGLISH:
### Antonyms:
- Smooth  
- Unmarked  
- Plain  
- Blank  
- Unetched

TIV:
### Asember a a lu kposo kposo la:
- Lue-lue
- I ver ikyav sha mi ga
- Wang
- Gbilin
- I nger kwagh sha mi ga

Source row: 908

ENGLISH:
But down signifies that you should draw the bow from the forefinger, up means that you should press the bow with the mouth(which is where your hand holds the bow).

TIV:
Kpa, shin tesen er we u tsua ivian sha ihôwegh I hiihii, sha tesen er we u kiigh adagh vea zwa, ka  u a lu ape wegh wou ku kôr adagh la.


In [13]:
# Sumary Counts

print("===== H Y P A  A U D I T =====")
print("Total candidates:", len(hypa_en_tiv_audit))

for column, label in [
    ("missing_text", "Missing or empty text"),
    ("duplicate_pair", "Duplicate English-Tiv pairs"),
    ("duplicate_english_input", "Repeated English inputs"),
    ("format_warning", "Formatting warnings"),
]:
    print(f"{label}: {int(hypa_en_tiv_audit[column].sum())}")

# Include every structural warning and repeated English input.
review_mask = (
    hypa_en_tiv_audit["missing_text"]
    | hypa_en_tiv_audit["duplicate_pair"]
    | hypa_en_tiv_audit["duplicate_english_input"]
    | hypa_en_tiv_audit["format_warning"]
)

flagged_records = hypa_en_tiv_audit.loc[
    review_mask,
    [
        "source_row_index",
        "english",
        "tiv",
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "format_warning",
    ],
]

print("\nTotal records requiring structural review:", len(flagged_records))
display(flagged_records)


===== H Y P A  A U D I T =====
Total candidates: 103
Missing or empty text: 0
Duplicate English-Tiv pairs: 0
Repeated English inputs: 0
Formatting warnings: 0

Total records requiring structural review: 0


,source_row_index,english,tiv,missing_text,duplicate_pair,duplicate_english_input,format_warning


In [ ]:
#MT560   AUDIT

In [15]:

import pandas as pd

# Preserve the original dataset row index for traceability.
mt560_audit = mt560_df[["eng", "tiv"]].copy()
mt560_audit.insert(
    0, "source_row_index", mt560_audit.index
)

mt560_audit = mt560_audit.rename(
    columns={"eng": "english", "tiv": "tiv"}
).reset_index(drop=True)

# Detect genuinely missing or empty values before normalizing text.
english_text = mt560_audit["english"].astype("string")
tiv_text = mt560_audit["tiv"].astype("string")

mt560_audit["missing_text"] = (
    english_text.isna()
    | english_text.str.strip().eq("").fillna(True)
    | tiv_text.isna()
    | tiv_text.str.strip().eq("").fillna(True)
)

# Use the same normalization function as the Hypa audit.
mt560_audit["english_normalized"] = (
    english_text.fillna("").map(normalize_for_comparison)
)
mt560_audit["tiv_normalized"] = (
    tiv_text.fillna("").map(normalize_for_comparison)
)

# Count repeated pairs and English inputs after normalization.
mt560_audit["duplicate_pair"] = mt560_audit.duplicated(
    subset=["english_normalized", "tiv_normalized"],
    keep="first",
)

mt560_audit["duplicate_english_input"] = mt560_audit.duplicated(
    subset=["english_normalized"],
    keep="first",
)

# Apply the same formatting-warning patterns used for Hypa.
format_pattern = (
    r"^\s*(?:SYSTEM:|USER:|ASSISTANT:)"
    r"|<\|[^|]+\|>|^\s*```"
)

mt560_audit["format_warning"] = (
    english_text.str.contains(
        format_pattern, case=False, regex=True, na=False
    )
    | tiv_text.str.contains(
        format_pattern, case=False, regex=True, na=False
    )
)

# Report counts using the same definitions as the Hypa audit.
print("===== MT560 AUDIT SUMMARY =====")
print("Total records:", len(mt560_audit))

for column, label in [
    ("missing_text", "Missing or empty text"),
    ("duplicate_pair", "Duplicate English-Tiv pairs"),
    ("duplicate_english_input", "Repeated English inputs"),
    ("format_warning", "Formatting warnings"),
]:
    print(f"{label}: {int(mt560_audit[column].sum())}")

# Add character lengths for comparison with Hypa.
mt560_audit["english_chars"] = english_text.fillna("").str.strip().str.len()
mt560_audit["tiv_chars"] = tiv_text.fillna("").str.strip().str.len()

print("\n===== TEXT LENGTH STATISTICS =====")
display(
    mt560_audit[["english_chars", "tiv_chars"]]
    .describe()
    .round(2)
)

# Include records flagged by any of the four checks.
review_mask = mt560_audit[
    [
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "format_warning",
    ]
].any(axis=1)

flagged_mt560 = mt560_audit.loc[
    review_mask,
    [
        "source_row_index",
        "english",
        "tiv",
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "format_warning",
    ],
]

print("\n===== STRUCTURAL REVIEW =====")
print("Flagged rows:", len(flagged_mt560))
display(flagged_mt560.head(30))

print("\n===== DUPLICATE PAIR EXAMPLES =====")
display(
    mt560_audit.loc[
        mt560_audit["duplicate_pair"],
        ["source_row_index", "english", "tiv"],
    ].head(10)
)

print("\n===== REPEATED ENGLISH INPUT EXAMPLES =====")
display(
    mt560_audit.loc[
        mt560_audit["duplicate_english_input"],
        ["source_row_index", "english", "tiv"],
    ].head(10)
)


===== MT560 AUDIT SUMMARY =====
Total records: 184113
Missing or empty text: 0
Duplicate English-Tiv pairs: 391
Repeated English inputs: 3116
Formatting warnings: 0

===== TEXT LENGTH STATISTICS =====


,english_chars,tiv_chars
count,184113.0,184113.0
mean,92.86,104.42
std,50.76,56.75
min,2.0,3.0
25%,56.0,63.0
50%,85.0,95.0
75%,120.0,135.0
max,569.0,544.0



===== STRUCTURAL REVIEW =====
Flagged rows: 3116


,source_row_index,english,tiv,missing_text,duplicate_pair,duplicate_english_input,format_warning
2090,2090,Of course not !,Ei he !,False,False,True,False
2788,2788,Of course not !,( Luka 6 : 12 - 14 ) Kape yange i lu la ga !,False,False,True,False
3117,3117,published by Jehovah 's Witnesses .,Ka Mbashiada mba Yehova ve gber u ye .,False,False,True,False
3232,3232,Be Zealous for True Worship,Lu A Gbashima Sha Kwagh U Mcivir U Mimi,False,True,True,False
3586,3586,[ Picture on page 27 ],[ Foto u sha peeji 27 ],False,False,True,False
4289,4289,"Study Articles 4 , 5 PAGES 24 - 32","Ngeren Mba Timen sha Mi 4 , 5 PEEJI 24 - 32",False,False,True,False
4511,4511,"Jesus said : "" No man can come to me unless th...","Yesu yange kaa ér : "" Ma or nana fatyô u gban ...",False,False,True,False
4527,4527,"[ Pictures on page 12 , 13 ]","[ Ufoto mba sha peeji 6 , 7 ]",False,False,True,False
5366,5366,What about you ?,We di yee ?,False,False,True,False
5369,5369,"If so , you are not alone .","Wea henen sha ve yô , ka we tseegh ga .",False,False,True,False



===== DUPLICATE PAIR EXAMPLES =====


,source_row_index,english,tiv
3232,3232,Be Zealous for True Worship,Lu A Gbashima Sha Kwagh U Mcivir U Mimi
12447,12447,"Receive God 's Spirit , Not the World 's","Ngohol Jijingi u Aôndo , I De Lu Jijingi u Tar..."
18445,18445,Let Jehovah Lead You to True Freedom,De Yehova A Hemen We A Za A We Ken Mlu U Ken U...
24542,24542,BY WAY OF REVIEW,U HIDEN SAVEN A SAV
25860,25860,Imitate Jesus - Teach With Love,Dondo Ikyav i Yesu sha u Tesen Ior sha Dooshima
28703,28703,Published by Jehovah 's Witnesses .,Ka Mbashiada mba Yehova ve gber u ye .
29108,29108,WHAT ABOUT YOU ?,WE YE ?
30989,30989,HOW WOULD YOU RESPOND ?,Ú NA MLUMUN WER NYI ?
32568,32568,Study Article 3 PAGES 13 - 17,Ngeren u Timen sha Mi 3 PEEJI 13 - 17
33839,33839,What can you do ?,Ú er nena ?



===== REPEATED ENGLISH INPUT EXAMPLES =====


,source_row_index,english,tiv
2090,2090,Of course not !,Ei he !
2788,2788,Of course not !,( Luka 6 : 12 - 14 ) Kape yange i lu la ga !
3117,3117,published by Jehovah 's Witnesses .,Ka Mbashiada mba Yehova ve gber u ye .
3232,3232,Be Zealous for True Worship,Lu A Gbashima Sha Kwagh U Mcivir U Mimi
3586,3586,[ Picture on page 27 ],[ Foto u sha peeji 27 ]
4289,4289,"Study Articles 4 , 5 PAGES 24 - 32","Ngeren Mba Timen sha Mi 4 , 5 PEEJI 24 - 32"
4511,4511,"Jesus said : "" No man can come to me unless th...","Yesu yange kaa ér : "" Ma or nana fatyô u gban ..."
4527,4527,"[ Pictures on page 12 , 13 ]","[ Ufoto mba sha peeji 6 , 7 ]"
5366,5366,What about you ?,We di yee ?
5369,5369,"If so , you are not alone .","Wea henen sha ve yô , ka we tseegh ga ."


In [16]:

print("===== MT560 AUDIT SUMMARY =====")
print("Total records:", len(mt560_audit))

for column, label in [
    ("missing_text", "Missing or empty text"),
    ("duplicate_pair", "Duplicate English-Tiv pairs"),
    ("duplicate_english_input", "Repeated English inputs"),
    ("format_warning", "Formatting warnings"),
]:
    print(f"{label}: {int(mt560_audit[column].sum())}")

review_mask = mt560_audit[
    [
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "format_warning",
    ]
].any(axis=1)

print("Records flagged by structural checks:", int(review_mask.sum()))

print("\nExact normalized duplicate pairs:")
display(
    mt560_audit.loc[
        mt560_audit["duplicate_pair"],
        ["source_row_index", "english", "tiv"],
    ].head(20)
)

print("\nRepeated English inputs:")
display(
    mt560_audit.loc[
        mt560_audit["duplicate_english_input"],
        ["source_row_index", "english", "tiv"],
    ].head(20)
)


===== MT560 AUDIT SUMMARY =====
Total records: 184113
Missing or empty text: 0
Duplicate English-Tiv pairs: 391
Repeated English inputs: 3116
Formatting warnings: 0
Records flagged by structural checks: 3116

Exact normalized duplicate pairs:


,source_row_index,english,tiv
3232,3232,Be Zealous for True Worship,Lu A Gbashima Sha Kwagh U Mcivir U Mimi
12447,12447,"Receive God 's Spirit , Not the World 's","Ngohol Jijingi u Aôndo , I De Lu Jijingi u Tar..."
18445,18445,Let Jehovah Lead You to True Freedom,De Yehova A Hemen We A Za A We Ken Mlu U Ken U...
24542,24542,BY WAY OF REVIEW,U HIDEN SAVEN A SAV
25860,25860,Imitate Jesus - Teach With Love,Dondo Ikyav i Yesu sha u Tesen Ior sha Dooshima
28703,28703,Published by Jehovah 's Witnesses .,Ka Mbashiada mba Yehova ve gber u ye .
29108,29108,WHAT ABOUT YOU ?,WE YE ?
30989,30989,HOW WOULD YOU RESPOND ?,Ú NA MLUMUN WER NYI ?
32568,32568,Study Article 3 PAGES 13 - 17,Ngeren u Timen sha Mi 3 PEEJI 13 - 17
33839,33839,What can you do ?,Ú er nena ?



Repeated English inputs:


,source_row_index,english,tiv
2090,2090,Of course not !,Ei he !
2788,2788,Of course not !,( Luka 6 : 12 - 14 ) Kape yange i lu la ga !
3117,3117,published by Jehovah 's Witnesses .,Ka Mbashiada mba Yehova ve gber u ye .
3232,3232,Be Zealous for True Worship,Lu A Gbashima Sha Kwagh U Mcivir U Mimi
3586,3586,[ Picture on page 27 ],[ Foto u sha peeji 27 ]
4289,4289,"Study Articles 4 , 5 PAGES 24 - 32","Ngeren Mba Timen sha Mi 4 , 5 PEEJI 24 - 32"
4511,4511,"Jesus said : "" No man can come to me unless th...","Yesu yange kaa ér : "" Ma or nana fatyô u gban ..."
4527,4527,"[ Pictures on page 12 , 13 ]","[ Ufoto mba sha peeji 6 , 7 ]"
5366,5366,What about you ?,We di yee ?
5369,5369,"If so , you are not alone .","Wea henen sha ve yô , ka we tseegh ga ."


In [17]:

import json
from datetime import datetime, timezone
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi()

SOURCE_IDS = {
    "mt560": "michsethowusu/english-tiv_sentence-pairs_mt560",
    "hypa": "hypaai/Hypa-Text-10k",
}

def get_source_metadata(dataset_id):
    info = api.dataset_info(dataset_id, revision="main")
    card = info.card_data

    if hasattr(card, "to_dict"):
        card = card.to_dict()
    elif card is None:
        card = {}
    else:
        card = dict(card)

    return {
        "dataset_id": dataset_id,
        "requested_revision": "main",
        "resolved_commit_sha": info.sha,
        "declared_license": card.get("license"),
        "source_url": f"https://huggingface.co/datasets/{dataset_id}",
    }

mt560_metadata = get_source_metadata(SOURCE_IDS["mt560"])
hypa_metadata = get_source_metadata(SOURCE_IDS["hypa"])

manifest = {
    "experiment_id": EXPERIMENT_CONFIG["experiment_id"],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "English-to-Tiv translation",
    "source_language_code": "en",
    "target_language_code": "tiv",
    "split_ratios": {
        "train": EXPERIMENT_CONFIG["train_ratio"],
        "validation": EXPERIMENT_CONFIG["validation_ratio"],
        "test": EXPERIMENT_CONFIG["test_ratio"],
    },
    "random_seed": EXPERIMENT_CONFIG["random_seed"],
    "sources": {
        "mt560": {
            **mt560_metadata,
            "split": "train",
            "loaded_rows": len(mt560_df),
            "audit": {
                "missing_or_empty_text": int(
                    mt560_audit["missing_text"].sum()
                ),
                "duplicate_pairs": int(
                    mt560_audit["duplicate_pair"].sum()
                ),
                "repeated_english_inputs": int(
                    mt560_audit["duplicate_english_input"].sum()
                ),
                "formatting_warnings": int(
                    mt560_audit["format_warning"].sum()
                ),
            },
            "eligibility_status": "pending_source_review",
        },
        "hypa": {
            **hypa_metadata,
            "split": "train",
            "loaded_rows": len(hypa_ds),
            "explicit_en_to_tiv_candidates": len(
                hypa_en_tiv_audit
            ),
            "audit": {
                "missing_or_empty_text": int(
                    hypa_en_tiv_audit["missing_text"].sum()
                ),
                "duplicate_pairs": int(
                    hypa_en_tiv_audit["duplicate_pair"].sum()
                ),
                "repeated_english_inputs": int(
                    hypa_en_tiv_audit[
                        "duplicate_english_input"
                    ].sum()
                ),
                "formatting_warnings": int(
                    hypa_en_tiv_audit["format_warning"].sum()
                ),
            },
            "eligibility_status": "pending_linguistic_review",
        },
    },
    "combination_status": "not_combined",
    "split_status": "not_created",
    "notes": [
        "Structural checks do not establish translation accuracy.",
        "Original source row identifiers must be preserved.",
        "Repeated English inputs are not automatically excluded.",
        "Dataset revisions must be pinned for reproducibility.",
        "License metadata is declared by the dataset host and "
        "does not replace provenance review.",
    ],
}

output_path = Path("tiv_source_provenance_manifest.json")
output_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Manifest saved to:", output_path.resolve())
print("MT560 revision:", mt560_metadata["resolved_commit_sha"])
print("MT560 declared license:", mt560_metadata["declared_license"])
print("Hypa revision:", hypa_metadata["resolved_commit_sha"])
print("Hypa declared license:", hypa_metadata["declared_license"])
print("\n===== MANIFEST PREVIEW =====")
print(json.dumps(manifest, indent=2, ensure_ascii=False))


Manifest saved to: /kaggle/working/tiv_source_provenance_manifest.json
MT560 revision: 252074f0d4c82c0005099b5eb4043fddc43026f1
MT560 declared license: cc-by-4.0
Hypa revision: 7a442cf5b029fab79540b093c9ef9abae5e30e96
Hypa declared license: apache-2.0

===== MANIFEST PREVIEW =====
{
  "experiment_id": "tiv-translation-pilot-v1",
  "created_at_utc": "2026-10-10T19:42:30.916190+00:00",
  "task": "English-to-Tiv translation",
  "source_language_code": "en",
  "target_language_code": "tiv",
  "split_ratios": {
    "train": 0.75,
    "validation": 0.1,
    "test": 0.15
  },
  "random_seed": 42,
  "sources": {
    "mt560": {
      "dataset_id": "michsethowusu/english-tiv_sentence-pairs_mt560",
      "requested_revision": "main",
      "resolved_commit_sha": "252074f0d4c82c0005099b5eb4043fddc43026f1",
      "declared_license": "cc-by-4.0",
      "source_url": "https://huggingface.co/datasets/michsethowusu/english-tiv_sentence-pairs_mt560",
      "split": "train",
      "loaded_rows": 184113,


In [ ]:
# AUTAMATED QUALITY CHECK

In [19]:

# Build a consistent quality-control audit
import re
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/tiv_quality_review")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep only the fields needed for this audit.
# The original source DataFrames remain unchanged.
mt560_qc = mt560_audit[
    ["source_row_index", "english", "tiv"]
].copy()
mt560_qc["source"] = "mt560"

hypa_qc = hypa_en_tiv_audit[
    ["source_row_index", "english", "tiv"]
].copy()
hypa_qc["source"] = "hypa"

def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()

def page_references(text):
    """Extract page-number sequences only when a page label is present."""
    text = str(text)
    pattern = (
        r"\b(?:pages?|peeji)\s*"
        r"(\d+(?:\s*[,–-]\s*\d+)*)"
    )
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    return [
        re.sub(r"\s+", "", match).replace("–", "-")
        for match in matches
    ]

CHAT_ARTIFACT_PATTERN = (
    r"<\|[^|]+\|>"
    r"|^\s*(?:SYSTEM|USER|ASSISTANT)\s*:"
    r"|^\s*```"
)

def audit_source(df):
    result = df.copy()

    result["english_normalized"] = result["english"].map(normalize_text)
    result["tiv_normalized"] = result["tiv"].map(normalize_text)

    result["missing_text"] = (
        result["english_normalized"].eq("")
        | result["tiv_normalized"].eq("")
    )

    result["duplicate_pair"] = result.duplicated(
        ["english_normalized", "tiv_normalized"],
        keep="first",
    )

    result["duplicate_english_input"] = result.duplicated(
        ["english_normalized"],
        keep="first",
    )

    result["english_chars"] = result["english_normalized"].str.len()
    result["tiv_chars"] = result["tiv_normalized"].str.len()

    # Broad thresholds identify unusual cases for inspection;
    # they are not language-quality rules.
    result["unusual_length"] = (
        result["english_chars"].lt(10)
        | result["tiv_chars"].lt(10)
        | result["english_chars"].gt(1000)
        | result["tiv_chars"].gt(1000)
    )

    result["extreme_length_ratio"] = False
    nonempty = (
        result["english_chars"].gt(0)
        & result["tiv_chars"].gt(0)
    )

    ratio = (
        result.loc[nonempty, "tiv_chars"]
        / result.loc[nonempty, "english_chars"]
    )
    result.loc[nonempty, "extreme_length_ratio"] = (
        ratio.lt(0.2) | ratio.gt(5.0)
    )

    result["chat_artifact"] = (
        result["english"].astype("string").str.contains(
            CHAT_ARTIFACT_PATTERN,
            case=False,
            regex=True,
            na=False,
        )
        | result["tiv"].astype("string").str.contains(
            CHAT_ARTIFACT_PATTERN,
            case=False,
            regex=True,
            na=False,
        )
    )

    result["structured_content"] = (
        result["english"].astype("string").str.contains(
            r"(^\s*#{1,6}\s|^\s*[-*]\s|\n\s*[-*]\s)",
            regex=True,
            na=False,
        )
        | result["tiv"].astype("string").str.contains(
            r"(^\s*#{1,6}\s|^\s*[-*]\s|\n\s*[-*]\s)",
            regex=True,
            na=False,
        )
    )

    en_pages = result["english"].map(page_references)
    tiv_pages = result["tiv"].map(page_references)

    result["page_reference_mismatch"] = [
        bool(en and tiv and en != tiv)
        for en, tiv in zip(en_pages, tiv_pages)
    ]

    flag_columns = [
        "missing_text",
        "duplicate_pair",
        "duplicate_english_input",
        "unusual_length",
        "extreme_length_ratio",
        "chat_artifact",
        "structured_content",
        "page_reference_mismatch",
    ]

    result["review_reasons"] = result.apply(
        lambda row: "; ".join(
            column for column in flag_columns
            if bool(row[column])
        ),
        axis=1,
    )

    result["needs_review"] = result["review_reasons"].ne("")
    return result

mt560_qc = audit_source(mt560_qc)
hypa_qc = audit_source(hypa_qc)

print("Automated QC completed.")
print("MT560 rows:", len(mt560_qc))
print("Hypa explicit English-to-Tiv candidates:", len(hypa_qc))


/tmp/ipykernel_49/3072550588.py:109: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  result["english"].astype("string").str.contains(
/tmp/ipykernel_49/3072550588.py:114: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | result["tiv"].astype("string").str.contains(


Automated QC completed.
MT560 rows: 184113
Hypa explicit English-to-Tiv candidates: 103


/tmp/ipykernel_49/3072550588.py:109: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  result["english"].astype("string").str.contains(
/tmp/ipykernel_49/3072550588.py:114: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | result["tiv"].astype("string").str.contains(


In [20]:

#Summarize the findings by source
FLAG_COLUMNS = [
    "missing_text",
    "duplicate_pair",
    "duplicate_english_input",
    "unusual_length",
    "extreme_length_ratio",
    "chat_artifact",
    "structured_content",
    "page_reference_mismatch",
]

def make_summary(name, df):
    summary = {
        "source": name,
        "total_records": len(df),
        "records_flagged_any_reason": int(df["needs_review"].sum()),
    }

    for column in FLAG_COLUMNS:
        summary[column] = int(df[column].sum())

    return summary

qc_summary = pd.DataFrame([
    make_summary("MT560", mt560_qc),
    make_summary("Hypa explicit English-to-Tiv", hypa_qc),
])

display(qc_summary.T)

print("\n===== MT560: POTENTIAL PAGE-REFERENCE MISMATCHES =====")
display(
    mt560_qc.loc[
        mt560_qc["page_reference_mismatch"],
        [
            "source_row_index",
            "english",
            "tiv",
            "review_reasons",
        ],
    ].head(30)
)

print("\n===== MT560: CHAT ARTIFACTS =====")
display(
    mt560_qc.loc[
        mt560_qc["chat_artifact"],
        [
            "source_row_index",
            "english",
            "tiv",
            "review_reasons",
        ],
    ].head(20)
)

print("\n===== HYPA: ALL FLAGGED CANDIDATES =====")
display(
    hypa_qc.loc[
        hypa_qc["needs_review"],
        [
            "source_row_index",
            "english",
            "tiv",
            "review_reasons",
        ],
    ]
)


,0,1
source,MT560,Hypa explicit English-to-Tiv
total_records,184113,103
records_flagged_any_reason,4593,6
missing_text,0,0
duplicate_pair,391,0
duplicate_english_input,3116,0
unusual_length,517,1
extreme_length_ratio,102,0
chat_artifact,0,0
structured_content,778,5



===== MT560: POTENTIAL PAGE-REFERENCE MISMATCHES =====


,source_row_index,english,tiv,review_reasons
106,106,Witnesses of Jehovah worldwide agree . - See t...,( Yohane 17 : 4 ) Mbashiada mba Yehova sha tar...,page_reference_mismatch
810,810,"For the term "" Jonadab , "" see Jehovah 's Witn...","Wea soo u seer fan kwagh u ishember i "" Yonada...",page_reference_mismatch
875,875,[ Picture on page 27 ],[ Foto u sha peeji 25 ],page_reference_mismatch
877,877,[ Blurb on page 26 ],[ Mkaanem ma sha peeji 32 la ],page_reference_mismatch
980,980,"For a detailed discussion of "" the appointed t...","( Yeremia 25 : 11 , 12 ; Daniel 9 : 1 - 3 ) Al...",page_reference_mismatch
1801,1801,[ Pictures on page 26 ],[ Foto u sha peeji 18 ],page_reference_mismatch
1806,1806,"June 22 , 1952 , pages 3 - 4 .","u Jun 22 , 1952 , peeji 3 zan zan 4 .",page_reference_mismatch
2720,2720,[ Picture on page 12 ],[ Foto u sha peeji 11 ],page_reference_mismatch
2722,2722,[ Picture on page 9 ],[ Foto u sha peeji 20 ],page_reference_mismatch
2723,2723,[ Picture on page 18 ],[ Foto u sha peeji 30 ],page_reference_mismatch



===== MT560: CHAT ARTIFACTS =====


,source_row_index,english,tiv,review_reasons



===== HYPA: ALL FLAGGED CANDIDATES =====


,source_row_index,english,tiv,review_reasons
8,671,### Antonyms:\n- Smooth \n- Unmarked \n- Pla...,### Asember a a lu kposo kposo la:\n- Lue-lue\...,structured_content
56,5094,### Word:\nAccustomed \n\n### Definition:\n- ...,### Ishember:\nU hôghul iyol\n\n### Inja kwagh...,structured_content
62,6250,### Definition:\n- **Adjective**: Attached; ad...,### mpase:\n- **usorun kwagh**: I vaa; I wa u ...,structured_content
70,6867,Stirred,Umoughon a kwagh teghelee,unusual_length
75,7213,### Definition:\n- **Adjective:** Not heard; n...,### Inja kwagh:\n- **Mpase:** I ungwa ga; i un...,structured_content
97,9098,### Word:\nAstir\n\n### Definition:\n- **Adjec...,### Ishember:\nJighilii\n\n### Inja kwagh:\n- ...,structured_content


In [22]:
# Prepare the manual-review files

REVIEW_SEED = 42

REVIEW_COLUMNS = [
    "source",
    "source_row_index",
    "english",
    "tiv",
    "review_reasons",
    "translation_accurate",
    "meaning_complete",
    "tiv_naturalness",
    "format_acceptable",
    "reviewer_notes",
    "reviewer_id",
]

def prepare_review_file(df, output_path):
    review = df[
        [
            "source",
            "source_row_index",
            "english",
            "tiv",
            "review_reasons",
        ]
    ].copy()

    for column in REVIEW_COLUMNS[5:]:
        review[column] = ""

    review = review[REVIEW_COLUMNS]
    review.to_csv(output_path, index=False, encoding="utf-8-sig")
    return review

# Hypa is small enough to review in full.
hypa_review = prepare_review_file(
    hypa_qc,
    OUTPUT_DIR / "hypa_en_tiv_manual_review.csv",
)

# MT560: prioritize flagged examples, then sample unflagged records.
mt560_flagged = mt560_qc.loc[mt560_qc["needs_review"]].copy()
mt560_unflagged = mt560_qc.loc[~mt560_qc["needs_review"]].copy()

# Review a bounded number of flagged examples for the first pass.
flagged_sample = mt560_flagged.sample(
    n=min(30, len(mt560_flagged)),
    random_state=REVIEW_SEED,
) if len(mt560_flagged) else mt560_flagged.copy()

# Add a random sample of clean-looking records.
random_sample = mt560_unflagged.sample(
    n=min(70, len(mt560_unflagged)),
    random_state=REVIEW_SEED,
) if len(mt560_unflagged) else mt560_unflagged.copy()

mt560_sample = pd.concat(
    [flagged_sample, random_sample],
    ignore_index=True,
).drop_duplicates(subset=["source_row_index"])

mt560_review = prepare_review_file(
    mt560_sample,
    OUTPUT_DIR / "mt560_manual_review_sample.csv",
)

print("Review files created:")
print("Hypa records:", len(hypa_review))
print("MT560 sample records:", len(mt560_review))
print("Output directory:", OUTPUT_DIR)

display(mt560_review.head(10))
display(hypa_review.head(10))


Review files created:
Hypa records: 103
MT560 sample records: 100
Output directory: /kaggle/working/tiv_quality_review


,source,source_row_index,english,tiv,review_reasons,translation_accurate,meaning_complete,tiv_naturalness,format_acceptable,reviewer_notes,reviewer_id
0,mt560,139738,[ Box / Pictures on page 29 ],[ Ngeren u kiriki / Foto u sha peeji 11 ],page_reference_mismatch,,,,,,
1,mt560,127564,"Therefore advertise , advertise , advertise , ...","Sha nahan yô , yôô nen , yôô nen , yôô nen ang...",duplicate_pair; duplicate_english_input,,,,,,
2,mt560,83779,"* Many years after the birth of Enosh , a desc...","* Yange i mar Enoshi anyom nga kar imôngo yô ,...",structured_content,,,,,,
3,mt560,123405,AGE : 43,Anyom : 43,unusual_length,,,,,,
4,mt560,169334,[ Picture on page 13 ],[ Foto u sha peeji 9 ],duplicate_english_input; page_reference_mismatch,,,,,,
5,mt560,2352,10 / 1,Kwaghwan u Injaa U sha Kwagh u Kwav man Ivese ...,unusual_length; extreme_length_ratio,,,,,,
6,mt560,184096,"[ Pictures on page 12 , 13 ]","[ Ufoto mba sha peeji 12 , 13 ]",duplicate_english_input,,,,,,
7,mt560,89063,The Seed Appears,Vor la Va,unusual_length,,,,,,
8,mt560,42691,SEE PAGES 9 - 11 .,NENGE PEEJI 15 .,page_reference_mismatch,,,,,,
9,mt560,109246,PAGE 5 .,PEEJI 5 .,unusual_length,,,,,,


,source,source_row_index,english,tiv,review_reasons,translation_accurate,meaning_complete,tiv_naturalness,format_acceptable,reviewer_notes,reviewer_id
0,hypa,81,He was one of the defenders of the Rouen School.,Un kpa lu mom ken mbakuran mba Rouen School.,,,,,,,
1,hypa,265,"She has to keep begging him, Will you keep me ...","""Gba u una za hemen u zamber a na, u za hemen ...",,,,,,,
2,hypa,304,Many species are day-flying with metallic patt...,Atô a akaa a uma nga pulugh atetan man mba a m...,,,,,,,
3,hypa,416,"In the Netherlands, officers of arms do not ex...","Ken tar u Netherlands yô, mbatomov mba utya mb...",,,,,,,
4,hypa,436,He was also a member of Air Canada's board of ...,Shi yange lu or u ken mbahemenev mba Air Canada.,,,,,,,
5,hypa,521,Under the scoreboard was a refreshment stall w...,Yange ijiir i teen kwaghyaan lu vegher u shin ...,,,,,,,
6,hypa,542,Talbot himself turned down a move to Italian s...,Talbot iyol na yange venda u yemen ken vegher ...,,,,,,,
7,hypa,642,He became a wealthy landowner of several thous...,Yange hingir shagba or u lun a inya kpishi kum...,,,,,,,
8,hypa,671,### Antonyms:\n- Smooth \n- Unmarked \n- Pla...,### Asember a a lu kposo kposo la:\n- Lue-lue\...,structured_content,,,,,,
9,hypa,708,That would be very early for him to have been ...,Ngula man i lu tsuaa yum u wen lun or ngeren k...,,,,,,,


In [23]:
# Save the audit artifacts

# Keep all records, including those flagged for review.
mt560_qc.to_csv(
    OUTPUT_DIR / "mt560_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

hypa_qc.to_csv(
    OUTPUT_DIR / "hypa_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

qc_summary.to_csv(
    OUTPUT_DIR / "quality_audit_summary.csv",
    index=False,
)

audit_metadata = {
    "experiment_id": EXPERIMENT_CONFIG["experiment_id"],
    "random_seed": REVIEW_SEED,
    "source_revisions": {
        "mt560": "252074f0d4c82c0005099b5eb4043fddc43026f1",
        "hypa": "7a442cf5b029fab79540b093c9ef9abae5e30e96",
    },
    "audit_flags": FLAG_COLUMNS,
    "hypa_manual_review_rows": len(hypa_review),
    "mt560_manual_review_rows": len(mt560_review),
    "combination_status": "not_combined",
    "training_status": "not_started",
}

(OUTPUT_DIR / "audit_metadata.json").write_text(
    json.dumps(audit_metadata, indent=2),
    encoding="utf-8",
)

print("===== SAVED ARTIFACTS =====")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")


===== SAVED ARTIFACTS =====
audit_metadata.json: 580 bytes
hypa_en_tiv_manual_review.csv: 19,626 bytes
hypa_quality_audit.csv: 42,873 bytes
mt560_manual_review_sample.csv: 19,993 bytes
mt560_quality_audit.csv: 88,984,248 bytes
quality_audit_summary.csv: 291 bytes


In [25]:
# reviewsers READY CSV files

import pandas as pd
from pathlib import Path

REVIEW_DIR = Path("/kaggle/working/tiv_quality_review")
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_REVISIONS = {
    "mt560": "252074f0d4c82c0005099b5eb4043fddc43026f1",
    "hypa": "7a442cf5b029fab79540b093c9ef9abae5e30e96",
}

def make_review_items(df, source, source_revision):
    items = df.copy()

    items["source"] = source
    items["source_revision"] = source_revision
    items["experiment_id"] = EXPERIMENT_CONFIG["experiment_id"]
    items["source_language"] = "en"
    items["target_language"] = "tiv"

    items["review_item_id"] = items["source_row_index"].map(
        lambda index: (
            f"{source}:{source_revision[:8]}:{int(index)}"
        )
    )

    # Preserve audit reasons when available.
    if "review_reasons" not in items.columns:
        items["review_reasons"] = ""

    columns = [
        "review_item_id",
        "experiment_id",
        "source",
        "source_revision",
        "source_row_index",
        "source_language",
        "target_language",
        "english",
        "tiv",
        "review_reasons",
    ]

    return items[columns].drop_duplicates(
        subset=["review_item_id"]
    )

hypa_items = make_review_items(
    hypa_qc,
    "hypa",
    SOURCE_REVISIONS["hypa"],
)

mt560_items = make_review_items(
    stratified_sample,
    "mt560",
    SOURCE_REVISIONS["mt560"],
)

review_items = pd.concat(
    [hypa_items, mt560_items],
    ignore_index=True,
)

review_items.to_csv(
    REVIEW_DIR / "review_items.csv",
    index=False,
    encoding="utf-8-sig",
)

# A blank assignment template. Give each reviewer their own copy.
assignments = review_items[["review_item_id"]].copy()
assignments["reviewer_id"] = ""
assignments["translation_accurate"] = ""
assignments["meaning_complete"] = ""
assignments["tiv_naturalness"] = ""
assignments["format_acceptable"] = ""
assignments["overall_decision"] = ""
assignments["reviewer_notes"] = ""
assignments["reviewed_at"] = ""

assignments.to_csv(
    REVIEW_DIR / "review_assignments_template.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Review items:", len(review_items))
print("Hypa items:", len(hypa_items))
print("MT560 items:", len(mt560_items))
print("Saved files:")
print(REVIEW_DIR / "review_items.csv")
print(REVIEW_DIR / "review_assignments_template.csv")



NameError: name '_stratified_sample' is not defined

In [1]:
from pathlib import Path

print("=== Candidate Tiv and MT560 files in Kaggle ===")

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

patterns = [
    "*mt560*",
    "*tiv*",
    "*provenance*manifest*",
    "train.jsonl",
    "validation.jsonl",
    "test.jsonl",
    "split_metadata.json",
]

for root in roots:
    print(f"\nSearching: {root}")

    if not root.exists():
        print("Directory does not exist.")
        continue

    found = set()

    for pattern in patterns:
        found.update(root.rglob(pattern))

    if not found:
        print("No matching files found.")
        continue

    for path in sorted(found):
        if path.is_file():
            print(f"{path} ({path.stat().st_size:,} bytes)")
        elif path.is_dir():
            print(f"{path}/ [directory]")

=== Candidate Tiv and MT560 files in Kaggle ===

Searching: /kaggle/input
No matching files found.

Searching: /kaggle/working
No matching files found.


In [2]:
from datasets import load_dataset

DATASET_ID = "michsethowusu/english-tiv_sentence-pairs_mt560"
REVISION = "252074f0d4c82c0005099b5eb4043fddc43026f1"

print("Loading pinned MT560 revision...")
dataset = load_dataset(
    DATASET_ID,
    revision=REVISION,
    split="train",
)

print("\n=== Dataset verification ===")
print("Dataset:", DATASET_ID)
print("Requested revision:", REVISION)
print("Rows:", len(dataset))
print("Columns:", dataset.column_names)
print("Features:", dataset.features)

print("\n=== First three source records ===")
for index in range(min(3, len(dataset))):
    print(f"\nSource row index: {index}")
    print(dataset[index])

Loading pinned MT560 revision...


README.md:   0%|          | 0.00/843 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]


=== Dataset verification ===
Dataset: michsethowusu/english-tiv_sentence-pairs_mt560
Requested revision: 252074f0d4c82c0005099b5eb4043fddc43026f1
Rows: 184113
Columns: ['eng', 'tiv']
Features: {'eng': Value('string'), 'tiv': Value('string')}

=== First three source records ===

Source row index: 0
{'eng': 'Prove Yourselves Ready', 'tiv': 'Kohol Nen Iyol'}

Source row index: 1
{'eng': 'I Shall Certainly Sanctify My Great Name', 'tiv': 'Me Tsegha Iti Yam i Vesen'}

Source row index: 2
{'eng': 'He Is Useful to Me', 'tiv': 'Ngum a Inja Kpishi Je'}


In [4]:
import json
import zipfile
from pathlib import Path

DATASET_ID = "michsethowusu/english-tiv_sentence-pairs_mt560"
REVISION = "252074f0d4c82c0005099b5eb4043fddc43026f1"
EXPECTED_ROWS = 184_113

assert len(dataset) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows, got {len(dataset)}"
)
assert set(dataset.column_names) == {"eng", "tiv"}

export_dir = Path("/kaggle/working/tiv_mt560_export")
export_dir.mkdir(parents=True, exist_ok=True)

canonical_path = export_dir / "mt560_canonical.jsonl"
provenance_path = export_dir / "mt560_row_provenance.jsonl"

with (
    canonical_path.open("w", encoding="utf-8") as canonical_file,
    provenance_path.open("w", encoding="utf-8") as provenance_file,
):
    for row_index, row in enumerate(dataset):
        record_id = f"mt560-{row_index:06d}"

        # Foundry's canonical DatasetRecord schema.
        canonical_record = {
            "id": record_id,
            "input": row["eng"],
            "target": row["tiv"],
            "language_code": "tiv",
        }

        # Separate sidecar preserves the original source location.
        provenance_record = {
            "record_id": record_id,
            "source_row_index": row_index,
            "dataset_id": DATASET_ID,
            "source_revision": REVISION,
            "source_split": "train",
            "source_input_column": "eng",
            "source_target_column": "tiv",
        }

        canonical_file.write(
            json.dumps(canonical_record, ensure_ascii=False) + "\n"
        )
        provenance_file.write(
            json.dumps(provenance_record, ensure_ascii=False) + "\n"
        )

# Package both files for transfer to Fedora.
archive_path = Path("/kaggle/working/tiv_mt560_export.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    archive.write(canonical_path, arcname=canonical_path.name)
    archive.write(provenance_path, arcname=provenance_path.name)

print("Export completed.")
print("Source rows:", len(dataset))
print("Canonical records:", sum(1 for _ in canonical_path.open(encoding="utf-8")))
print("Provenance records:", sum(1 for _ in provenance_path.open(encoding="utf-8")))
print("Archive:", archive_path)
print("Archive size (MB):", round(archive_path.stat().st_size / 1_000_000, 2))
print("\nFirst canonical record:")
print(canonical_path.open(encoding="utf-8").readline().strip())
print("\nFirst provenance record:")
print(provenance_path.open(encoding="utf-8").readline().strip())

Export completed.
Source rows: 184113
Canonical records: 184113
Provenance records: 184113
Archive: /kaggle/working/tiv_mt560_export.zip
Archive size (MB): 17.11

First canonical record:
{"id": "mt560-000000", "input": "Prove Yourselves Ready", "target": "Kohol Nen Iyol", "language_code": "tiv"}

First provenance record:
{"record_id": "mt560-000000", "source_row_index": 0, "dataset_id": "michsethowusu/english-tiv_sentence-pairs_mt560", "source_revision": "252074f0d4c82c0005099b5eb4043fddc43026f1", "source_split": "train", "source_input_column": "eng", "source_target_column": "tiv"}
